> **데이터셋 안내** — 이 노트북이 참조하는 HF 데이터셋은 공개 배포하지 않는다.
> AI Hub 원본에서 재생성하는 절차는 [docs/data/data-pipeline.md](../docs/data/data-pipeline.md)「가공 데이터셋은 배포하지 않는다 — 재현 경로」에 있다.

In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
import random
from numpy.typing import NDArray

load_dotenv()
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

import torch
from transformers import AutoTokenizer
from datasets import load_dataset

In [2]:
# Config
config = {
    "num_labels": 188,
    "seed": 42,
    "model_name": "klue/roberta-large",
    "rev": "28d911204e9022eda172571ca8cc61eaffd942f7",
    "dataset": "ingyoun/patent-clean-text",
    "repo": "ingyoun/patent-clean-text-roberta-tokenized",
}

In [3]:
random.seed(config["seed"])
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])
torch.cuda.manual_seed_all(config["seed"])

In [4]:
dataset = load_dataset(config["dataset"])

In [5]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=config["rev"])

In [6]:
print(f"Vocab Size: {tokenizer.vocab_size}")
print(f"Max Length: {tokenizer.model_max_length}")
print(f"Max Length: {tokenizer.all_special_tokens}")

Vocab Size: 32000
Max Length: 512
Max Length: ['[CLS]', '[SEP]', '[UNK]', '[PAD]', '[MASK]']


- roberta의 max length는 bert와 동일하게 512지만 vocab size는 32,000으로 bert의 8,002에 비해 길다

In [7]:
def build_inputs(ex):
    inputs = dict()
    fields = ["invention_title", "ipc_main", "abstract", "claims"]
    text = " ".join(str(ex[field]) for field in fields if ex[field])   # 빈 필드 skip, 개행/들여쓰기 없음
    return {
        "document_id": ex["document_id"],
        "input" : text,
        "label_ids": ex["label_ids"],
        "kobert_len": ex["kobert_len"],
        "length_bin": ex["length_bin"],
    }

In [ ]:
dataset_inputs = dataset.map(build_inputs, remove_columns=dataset["train"].column_names)

In [10]:
dataset_inputs

DatasetDict({
    train: Dataset({
        features: ['document_id', 'label_ids', 'kobert_len', 'length_bin', 'input'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'label_ids', 'kobert_len', 'length_bin', 'input'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'label_ids', 'kobert_len', 'length_bin', 'input'],
        num_rows: 11162
    })
})

In [13]:
def to_features(ex):
    out = tokenizer(ex["input"])
    batch_size = len(ex["input"])
    y = np.zeros((batch_size, config["num_labels"]), dtype=np.float32)
    for i, ids in enumerate(ex["label_ids"]):
        y[i, ids] = 1.0
    out["labels"] = y.tolist()
    out["kobert_len"] = ex["kobert_len"]
    out["length_bin"] = ex["length_bin"]
    return out

In [32]:
remove_cols = [c for c in dataset_inputs["train"].column_names if c not in ["document_id", "kobert_len", "length_bin"]]

ds_tok = dataset_inputs.map(
    to_features, 
    remove_columns=remove_cols,
    batched=True
    )

- kobert, modernbert와 달리 'token_type_ids'가 존재

In [10]:
len(ds_tok["train"][0]["input_ids"]), ds_tok["train"][0]["kobert_len"]

(789, 856)

In [12]:
def add_length(batch):
    batch["length"] = [len(ids) for ids in batch["input_ids"]]
    batch["diff"] = [
        kb - ln for kb, ln in zip(batch["kobert_len"], batch["length"])
    ]
    return batch

In [ ]:
ds_tok = ds_tok.map(add_length, batched=True, num_proc=4)

In [37]:
train_lengths = np.array(ds_tok["train"]["length"])
val_lengths = np.array(ds_tok["val"]["length"])
test_lengths = np.array(ds_tok["test"]["length"])

print(f"train length mean={train_lengths.mean():.1f}, median={np.median(train_lengths):.0f}")
print(f"val length mean={val_lengths.mean():.1f}, median={np.median(val_lengths):.0f}")
print(f"test length mean={test_lengths.mean():.1f}, median={np.median(test_lengths):.0f}")

train length mean=785.0, median=618
val length mean=785.5, median=620
test length mean=793.9, median=624


In [38]:
train_diff = np.array(ds_tok["train"]["diff"])
val_diff = np.array(ds_tok["val"]["diff"])
test_diff = np.array(ds_tok["test"]["diff"])

print(f"train diff mean={train_diff.mean():.1f}, median={np.median(train_diff):.0f}")
print(f"val diff mean={val_diff.mean():.1f}, median={np.median(val_diff):.0f}")
print(f"test diff mean={test_diff.mean():.1f}, median={np.median(test_diff):.0f}")

train diff mean=98.0, median=73
val diff mean=96.0, median=73
test diff mean=98.6, median=74


In [39]:
def length_percentile(lengths: NDArray):
    for p in [50, 75, 90, 95, 99]:
        print(f"p{p} = {np.percentile(lengths, p):.0f}")
    print(f"max={lengths.max()}, >512 비율={np.mean(lengths > 512):.2%}")


print(f"Train Token Length :")
length_percentile(train_lengths)
print(f"\nVal Token Length : ")
length_percentile(val_lengths)
print(f"\nTest Token Length : ")
length_percentile(test_lengths)

Train Token Length :
p50 = 618
p75 = 913
p90 = 1354
p95 = 1787
p99 = 3620
max=10275, >512 비율=63.51%

Val Token Length : 
p50 = 620
p75 = 907
p90 = 1335
p95 = 1761
p99 = 3716
max=8882, >512 비율=63.45%

Test Token Length : 
p50 = 624
p75 = 921
p90 = 1368
p95 = 1782
p99 = 3790
max=8951, >512 비율=64.22%


In [41]:
ds_tok

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'length', 'diff'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'length', 'diff'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'length', 'diff'],
        num_rows: 11162
    })
})

In [42]:
ds_tok = ds_tok.remove_columns(["length", "diff", "token_type_ids"])
ds_tok

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

In [ ]:
ds_tok.push_to_hub(config["repo"])